# Study 1 — 01e: Hypothesis Recycling Analysis

Quantifies hypothesis perseveration — the tendency for the model to re-propose previously tested hypotheses. Uses `hypo_status` (novel/revised/repeated) and `hypo_antecedent_sid`.

**Outputs:**
- `outputs/study1_analysis/tables/hypo_status_distribution.csv`
- `outputs/study1_analysis/tables/repetition_by_position.csv`
- `outputs/study1_analysis/tables/repetition_chains.csv`
- `outputs/study1_analysis/figures/hypo_status_by_task.png`
- `outputs/study1_analysis/figures/repetition_by_position.png`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
df = build_sentence_df(traces)
hypo_df = df[df['micro_label'] == 'HYPO'].copy()
print(f'Total HYPO sentences: {len(hypo_df):,}')

## 5a. hypo_status Distribution

In [ ]:
section_header('5a. hypo_status Distribution')

# Overall
status_counts = hypo_df['hypo_status'].value_counts()
status_pct = (status_counts / len(hypo_df) * 100).round(1)
print('Overall hypo_status:')
for k in ['novel', 'revised', 'repeated']:
    v = status_counts.get(k, 0)
    p = status_pct.get(k, 0)
    print(f'  {k}: {v} ({p}%)')

# Per task
task_status = hypo_df.groupby(['task_id', 'hypo_status']).size().unstack(fill_value=0)
task_status_pct = task_status.div(task_status.sum(axis=1), axis=0) * 100
print('\nhypo_status by task (%):')
display(task_status_pct.round(1))

# Per set
set_status = hypo_df.groupby(['set', 'hypo_status']).size().unstack(fill_value=0)
set_status_pct = set_status.div(set_status.sum(axis=1), axis=0) * 100
print('\nhypo_status by set (%):')
display(set_status_pct.round(1))

# Per completion status
comp_status = hypo_df.groupby(['completed', 'hypo_status']).size().unstack(fill_value=0)
comp_status_pct = comp_status.div(comp_status.sum(axis=1), axis=0) * 100
comp_status_pct.index = ['Truncated', 'Completed']
print('\nhypo_status by completion status (%):')
display(comp_status_pct.round(1))

# Save
save_table(task_status_pct, 'hypo_status_distribution.csv')

In [ ]:
# Visualise: stacked bar chart by task
fig, ax = plt.subplots(figsize=(8, 5))
status_order = ['novel', 'revised', 'repeated']
colours = {'novel': '#55A868', 'revised': '#4C72B0', 'repeated': '#C44E52'}
bottom = np.zeros(len(task_status_pct))
for status in status_order:
    if status in task_status_pct.columns:
        vals = task_status_pct[status].values
        ax.bar(task_status_pct.index, vals, bottom=bottom, label=status,
               color=colours[status], edgecolor='white')
        bottom += vals

ax.set_xlabel('Task')
ax.set_ylabel('Percentage of HYPO sentences')
ax.set_title('Hypothesis Status by Task')
ax.legend(title='hypo_status')
ax.set_ylim(0, 105)
plt.tight_layout()
save_fig(fig, 'hypo_status_by_task.png')
plt.show()

## 5b. Repetition Rate by Trace Position

In [ ]:
section_header('5b. Repetition Rate by Trace Position')

pos_status = hypo_df.groupby('position_third')['hypo_status'].value_counts().unstack(fill_value=0)
pos_pct = pos_status.div(pos_status.sum(axis=1), axis=0) * 100

print('hypo_status by position third (%):')
display(pos_pct.round(1))

# Visualise: line chart
fig, ax = plt.subplots(figsize=(8, 5))
for status in ['novel', 'revised', 'repeated']:
    if status in pos_pct.columns:
        ax.plot(['early', 'middle', 'late'], pos_pct[status].values,
                marker='o', linewidth=2, label=status, color=colours[status])

ax.set_xlabel('Position in trace')
ax.set_ylabel('Percentage of HYPO sentences')
ax.set_title('Hypothesis Status by Trace Position')
ax.legend(title='hypo_status')
plt.tight_layout()
save_fig(fig, 'repetition_by_position.png')
plt.show()

save_table(pos_pct, 'repetition_by_position.csv')

## 5c. Repetition Chains

In [ ]:
section_header('5c. Repetition Chains')

def find_repetition_chains(trace_hypos):
    """
    Find chains of revised/repeated hypotheses using hypo_antecedent_sid.
    A chain is a sequence where each HYPO points back to a prior HYPO.
    """
    trace_hypos = trace_hypos.sort_values('sentence_id')
    # Build antecedent graph: sid → antecedent_sid
    antecedent = {}
    for _, row in trace_hypos.iterrows():
        if pd.notna(row['hypo_antecedent_sid']):
            antecedent[row['sentence_id']] = int(row['hypo_antecedent_sid'])

    # Find roots (HYPOs with no antecedent, or whose antecedent is not in this set)
    all_sids = set(trace_hypos['sentence_id'])
    roots = [sid for sid in all_sids if sid not in antecedent or antecedent[sid] not in all_sids]

    # Trace chains forward
    children = {}
    for sid, ant in antecedent.items():
        if ant in all_sids:
            children.setdefault(ant, []).append(sid)

    chains = []
    for root in roots:
        chain = [root]
        current = root
        while current in children:
            nexts = sorted(children[current])
            current = nexts[0]  # follow first child
            chain.append(current)
            # Mark other children as separate chain starts
            for other in nexts[1:]:
                sub_chain = [other]
                sub_current = other
                while sub_current in children:
                    sub_nexts = sorted(children[sub_current])
                    sub_current = sub_nexts[0]
                    sub_chain.append(sub_current)
                if len(sub_chain) > 1:
                    chains.append(sub_chain)
        if len(chain) > 1:
            chains.append(chain)
    return chains

# Find chains across all traces
chain_data = []
for key, grp in df[df['micro_label'] == 'HYPO'].groupby('trace_key'):
    task_id = grp['task_id'].iloc[0]
    chains = find_repetition_chains(grp)
    for chain in chains:
        chain_data.append({
            'trace_key': key,
            'task_id': task_id,
            'chain_length': len(chain),
            'chain_sids': chain,
        })

chain_df = pd.DataFrame(chain_data)
print(f'Total repetition chains: {len(chain_df)}')

if len(chain_df) > 0:
    # Longest chain per trace
    max_chain_per_trace = chain_df.groupby('trace_key')['chain_length'].max()
    print(f'Max chain length per trace: mean={max_chain_per_trace.mean():.1f}, max={max_chain_per_trace.max()}')

    # Perseverative traces: ≥5 repetitions in same thread
    perseverative = max_chain_per_trace[max_chain_per_trace >= 5]
    n_persev = len(perseverative)
    pct_persev = n_persev / df.trace_key.nunique() * 100
    print(f'\nPerseverative traces (≥5 in same thread): {n_persev}/{df.trace_key.nunique()} ({pct_persev:.1f}%)')

    # By task
    if 'task_id' in chain_df.columns:
        task_chains = chain_df.groupby('task_id').agg(
            n_chains=('chain_length', 'size'),
            mean_length=('chain_length', 'mean'),
            max_length=('chain_length', 'max'),
        )
        print('\nRepetition chains by task:')
        display(task_chains.round(1))

save_table(chain_df[['trace_key', 'task_id', 'chain_length']] if len(chain_df) > 0 else pd.DataFrame(),
           'repetition_chains.csv')

## 5d. Revised vs Repeated: Semantic Distance

> Note: If sentence embeddings are not precomputed and available, this section reports the classification thresholds from the PHASE3 log rather than recomputing embeddings.

In [ ]:
section_header('5d. Revised vs Repeated')

# The hypo_status classification was done with sentence-transformers (all-MiniLM-L6-v2)
# Thresholds: repeated >= 0.90, revised >= 0.70
# We report the classification breakdown rather than recomputing embeddings here.

print('Classification thresholds (from compute_hypo_status.py):')
print('  repeated: cosine similarity >= 0.90 (near-identical)')
print('  revised:  cosine similarity >= 0.70, < 0.90 (same dimension, different specifics)')
print('  novel:    cosine similarity < 0.70 (new feature dimension)')
print()
print('By definition:')
print('  - "repeated" hypotheses have mean cosine similarity ≥ 0.90 with their antecedent')
print('  - "revised" hypotheses have mean cosine similarity in [0.70, 0.90)')
print('  - The gap between revised and repeated (0.20 cosine units) represents')
print('    the difference between genuine refinement and near-verbatim restating.')

## 5e. Relationship Between Recycling and Trace Outcome

In [ ]:
section_header('5e. Recycling and Trace Outcome')

# For completed traces with RULE: trace back from the final accepted hypothesis
completed_df = df[df['completed'] == True]
rule_traces = completed_df[completed_df['micro_label'] == 'RULE']['trace_key'].unique()
print(f'Completed traces with RULE: {len(rule_traces)}')

# For each RULE trace, find the JUDGE that the RULE depends on,
# then find the HYPO that the JUDGE depends on, and check its hypo_status
results = []
for key in rule_traces:
    trace = df[df['trace_key'] == key].sort_values('sentence_id')
    rules = trace[trace['micro_label'] == 'RULE']
    if len(rules) == 0:
        continue
    last_rule = rules.iloc[-1]

    # Find the JUDGE in RULE's depends_on
    rule_deps = last_rule['depends_on']
    judge_sids = trace.loc[
        (trace['sentence_id'].isin(rule_deps)) & (trace['micro_label'] == 'JUDGE'),
        'sentence_id'
    ].tolist()

    if not judge_sids:
        results.append({'trace_key': key, 'source': 'unknown'})
        continue

    # Find the HYPO that this JUDGE depends on
    judge_row = trace[trace['sentence_id'] == judge_sids[0]].iloc[0]
    judge_deps = judge_row['depends_on']
    hypo_sids = trace.loc[
        (trace['sentence_id'].isin(judge_deps)) & (trace['micro_label'] == 'HYPO'),
        'sentence_id'
    ].tolist()

    if not hypo_sids:
        results.append({'trace_key': key, 'source': 'unknown'})
        continue

    hypo_row = trace[trace['sentence_id'] == hypo_sids[0]].iloc[0]
    status = hypo_row.get('hypo_status', 'unknown')
    results.append({'trace_key': key, 'source': status if pd.notna(status) else 'unknown'})

results_df = pd.DataFrame(results)
source_counts = results_df['source'].value_counts()
source_pct = (source_counts / len(results_df) * 100).round(1)

print('\nFinal rule hypothesis source:')
for k, v in source_counts.items():
    print(f'  {k}: {v} ({source_pct[k]}%)')

novel_pct = source_pct.get('novel', 0)
non_novel_pct = source_pct.get('revised', 0) + source_pct.get('repeated', 0)
print(f'\nFrom novel HYPO: {novel_pct}%')
print(f'From revised/repeated HYPO: {non_novel_pct}%')